# NSFW Adversarial Attack Demo

pNSFWMedia分類器に対する敵対的攻撃のデモンストレーション。

## 準備
- ターゲットモデル: `models/target_classifier/pnsfwmedia_classifier.keras`
- 埋め込みデータ: `dataset/nsfw_embeddings/` (.npy形式)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import (
    load_target_model, load_embeddings, predict_batch,
    compute_metrics, set_seed, detect_gpu, setup_logging
)
from src.attacks.fgsm import FGSMAttack, FGSMConfig
from src.attacks.pgd import PGDAttack, PGDConfig
from src.attacks.cw import CWAttack, CWConfig
from src.attacks.deepfool import DeepFoolAttack, DeepFoolConfig

setup_logging('INFO')
set_seed(42)
detect_gpu()

print('TensorFlow version:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))

## 1. モデルと埋め込みの読み込み

In [ ]:
# モデルの読み込み
model = load_target_model('../models/target_classifier/pnsfwmedia_classifier.keras')
model.summary()

# 埋め込みの読み込み
embeddings = load_embeddings('../dataset/nsfw_embeddings')
print(f'Embeddings shape: {embeddings.shape}')

# 元の予測を確認
original_probs = predict_batch(model, embeddings)
nsfw_mask = original_probs >= 0.5
print(f'NSFW samples: {nsfw_mask.sum()}/{len(original_probs)}')
print(f'Mean NSFW probability: {original_probs[nsfw_mask].mean():.4f}')

## 2. 元の予測分布

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(original_probs, bins=50, alpha=0.7, edgecolor='black')
ax.axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Threshold (0.5)')
ax.set_xlabel('NSFW Probability', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Original NSFW Probability Distribution', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 3. FGSM攻撃

In [ ]:
# 複数のepsilon値で実験
epsilons = [0.01, 0.03, 0.05, 0.07, 0.1]
fgsm_results = {}

for eps in epsilons:
    config = FGSMConfig(epsilon=eps)
    attack = FGSMAttack(model, config)
    adv, noise = attack.attack(embeddings)
    adv_probs = predict_batch(model, adv)
    metrics = compute_metrics(original_probs, adv_probs, noise)
    fgsm_results[eps] = {
        'metrics': metrics,
        'adv_probs': adv_probs,
    }
    sr = metrics.get('success_rate_0.5', 0)
    print(f'FGSM eps={eps:.2f}: SR@0.5={sr:.2%}, '
          f'Avg reduction={metrics.get("avg_prob_reduction", 0):.4f}')

In [ ]:
# FGSM: epsilon vs success rate
fig, ax = plt.subplots(figsize=(10, 5))
for threshold in [0.5, 0.4, 0.3]:
    rates = [fgsm_results[e]['metrics'].get(f'success_rate_{threshold}', 0) for e in epsilons]
    ax.plot(epsilons, rates, 'o-', label=f'Threshold={threshold}', linewidth=2)

ax.set_xlabel('Epsilon (ε)', fontsize=12)
ax.set_ylabel('Success Rate', fontsize=12)
ax.set_title('FGSM: Attack Success Rate vs Epsilon', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

## 4. PGD攻撃

In [ ]:
pgd_results = {}

for eps in epsilons:
    config = PGDConfig(epsilon=eps, alpha=eps/4, iterations=20)
    attack = PGDAttack(model, config)
    adv, noise, iters = attack.attack(embeddings)
    adv_probs = predict_batch(model, adv)
    metrics = compute_metrics(original_probs, adv_probs, noise)
    metrics['avg_iterations'] = float(iters.mean())
    pgd_results[eps] = {
        'metrics': metrics,
        'adv_probs': adv_probs,
    }
    sr = metrics.get('success_rate_0.5', 0)
    print(f'PGD eps={eps:.2f}: SR@0.5={sr:.2%}, '
          f'Avg iters={metrics["avg_iterations"]:.1f}')

In [ ]:
# FGSM vs PGD comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for method_name, results in [('FGSM', fgsm_results), ('PGD', pgd_results)]:
    rates_05 = [results[e]['metrics'].get('success_rate_0.5', 0) for e in epsilons]
    ax1.plot(epsilons, rates_05, 'o-', label=method_name, linewidth=2)

ax1.set_xlabel('Epsilon', fontsize=12)
ax1.set_ylabel('Success Rate @ 0.5', fontsize=12)
ax1.set_title('FGSM vs PGD: Success Rate', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Probability distributions comparison
best_eps = 0.05
ax2.hist(original_probs, bins=50, alpha=0.5, label='Original', color='blue')
ax2.hist(fgsm_results[best_eps]['adv_probs'], bins=50, alpha=0.5, label=f'FGSM ε={best_eps}', color='orange')
ax2.hist(pgd_results[best_eps]['adv_probs'], bins=50, alpha=0.5, label=f'PGD ε={best_eps}', color='green')
ax2.axvline(x=0.5, color='red', linestyle='--', label='Threshold')
ax2.set_xlabel('NSFW Probability', fontsize=12)
ax2.set_ylabel('Count', fontsize=12)
ax2.set_title('Probability Distributions', fontsize=14)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

## 5. C&W攻撃

In [ ]:
# C&W attack (slower, use subset for demo)
demo_embeddings = embeddings[:100]
demo_probs = original_probs[:100]

config = CWConfig(c=1.0, kappa=0.0, iterations=500, batch_size=32)
attack = CWAttack(model, config)
adv, noise, iters = attack.attack(demo_embeddings)
adv_probs = predict_batch(model, adv)
metrics = compute_metrics(demo_probs, adv_probs, noise)

print(f'C&W: SR@0.5={metrics.get("success_rate_0.5", 0):.2%}')
print(f'Avg L2 norm: {metrics.get("avg_l2_norm", 0):.6f}')
print(f'Avg prob reduction: {metrics.get("avg_prob_reduction", 0):.4f}')

## 6. DeepFool攻撃

In [ ]:
# DeepFool (per-sample, use subset)
config = DeepFoolConfig(max_iterations=100, overshoot=0.02)
attack = DeepFoolAttack(model, config)
adv, noise, iters = attack.attack(demo_embeddings)
adv_probs = predict_batch(model, adv)
metrics = compute_metrics(demo_probs, adv_probs, noise)

print(f'DeepFool: SR@0.5={metrics.get("success_rate_0.5", 0):.2%}')
print(f'Avg L2 norm: {metrics.get("avg_l2_norm", 0):.6f}')
print(f'Avg iterations: {iters.mean():.1f}')

## 7. 全手法の比較サマリー

In [ ]:
# Summary table
print(f'{"Method":<12} {"SR@0.5":>8} {"SR@0.3":>8} {"Avg L2":>10} {"Avg ΔP":>10}')
print('-' * 52)

best_eps = 0.05
for name, res in [
    ('FGSM', fgsm_results[best_eps]['metrics']),
    ('PGD', pgd_results[best_eps]['metrics']),
]:
    print(f'{name:<12} '
          f'{res.get("success_rate_0.5", 0):>8.2%} '
          f'{res.get("success_rate_0.3", 0):>8.2%} '
          f'{res.get("avg_l2_norm", 0):>10.6f} '
          f'{res.get("avg_prob_reduction", 0):>10.4f}')